# FOOTSIES — Tabular PPO + GAE

In [1]:
import numpy as np
from pathlib import Path

# Keep the original 12-state encoding, 6 actions, and scripted opponent.
PLATFORM, PORT, LAUNCH_BINARIES = "mac", 50051, False
TRAIN_EPISODES, MAX_STEPS = 300, 600
EVAL_EVERY, QUICK_EVAL_EPISODES, FINAL_EVAL_EPISODES = 50, 30, 60
NUM_STATES, NUM_ACTIONS = 12, 6
DISTANCE_BINS = [0.07, 0.15, 0.30]
GUARD_REWARD = 0.10

# PPO + GAE hyperparameters.
GAMMA, GAE_LAMBDA = 0.98, 0.95
CLIP_EPS = 0.20
POLICY_LR, VALUE_LR = 0.5, 0.5
BATCH_EPISODES, PPO_EPOCHS, MINIBATCH_SIZE = 5, 4, 128
CHECKPOINT_DIR = Path("footsies_checkpoints")


def encode_state(obs):
    distance = int(np.digitize(float(obs[0]), DISTANCE_BINS))
    opp = np.asarray(obs)[-37:]
    if opp[32] > 0.5:
        phase = 2
    else:
        action = int(np.argmax(opp[8:25]))
        if action not in ATTACK_INDICES:
            phase = 0
        else:
            progress = float(opp[25]) / max(float(opp[26]), 1e-6)
            phase = 1 if progress < 0.65 else 2
    return distance * 3 + phase


def guard_health(obs, opponent=False):
    if opponent:
        return int(np.argmax(obs[55:59]))
    return int(np.argmax(obs[5:9]))


def opponent_action(obs, rng):
    d = float(obs[0])
    if d > 0.30:
        actions, probs = [2, 1, 5], [0.80, 0.10, 0.10]
    elif d > 0.15:
        actions, probs = [2, 5, 3, 1], [0.50, 0.25, 0.15, 0.10]
    elif d > 0.07:
        actions, probs = [3, 5, 1, 0], [0.45, 0.20, 0.20, 0.15]
    else:
        actions, probs = [3, 4, 1, 0], [0.55, 0.25, 0.15, 0.05]
    return int(rng.choice(actions, p=probs))

In [2]:
# Tabular actor and critic: no neural network or autograd needed.
logits = np.zeros((NUM_STATES, NUM_ACTIONS), dtype=np.float64)
values = np.zeros(NUM_STATES, dtype=np.float64)


def softmax(x):
    x = np.asarray(x, dtype=np.float64)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum(axis=-1, keepdims=True)


def choose_action(state, rng, greedy=False):
    probs = softmax(logits[state])
    return int(np.argmax(probs) if greedy else rng.choice(NUM_ACTIONS, p=probs))


def compute_gae(rewards, old_values, next_values, terminals):
    """Compute per-episode GAE and critic targets using the rollout critic."""
    rewards = np.asarray(rewards, dtype=np.float64)
    old_values = np.asarray(old_values, dtype=np.float64)
    next_values = np.asarray(next_values, dtype=np.float64)
    terminals = np.asarray(terminals, dtype=bool)

    advantages = np.zeros_like(rewards)
    gae = 0.0
    for t in reversed(range(len(rewards))):
        # Bootstrap at a time-limit truncation, but not at a true terminal state.
        nonterminal = 1.0 - float(terminals[t])
        delta = rewards[t] + GAMMA * nonterminal * next_values[t] - old_values[t]
        gae = delta + GAMMA * GAE_LAMBDA * nonterminal * gae
        advantages[t] = gae
    return advantages, advantages + old_values


def ppo_update(episodes, rng):
    """Run clipped PPO and critic regression on a fixed on-policy batch."""
    keys = ("states", "actions", "old_log_probs", "advantages", "returns")
    batch = {key: np.concatenate([ep[key] for ep in episodes]) for key in keys}
    states = batch["states"]
    actions = batch["actions"]
    old_log_probs = batch["old_log_probs"]
    targets = batch["returns"]
    adv = batch["advantages"].copy()
    n = len(states)

    # Standardize the advantage using statistics from the entire rollout batch.
    if adv.std() > 1e-8:
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    for _ in range(PPO_EPOCHS):
        for indices in np.array_split(rng.permutation(n),
                                      max(1, int(np.ceil(n / MINIBATCH_SIZE)))):
            s, a = states[indices], actions[indices]
            probs = softmax(logits[s])
            log_probs = np.log(np.maximum(probs[np.arange(len(s)), a], 1e-300))
            ratio = np.exp(log_probs - old_log_probs[indices])
            advantage = adv[indices]

            # The gradient of min(ratio*A, clip(ratio)*A) is zero
            # when positive A has ratio > 1+eps or negative A has ratio < 1-eps.
            active = ((advantage >= 0) & (ratio <= 1 + CLIP_EPS)) | (
                (advantage < 0) & (ratio >= 1 - CLIP_EPS)
            )
            coeff = np.where(active, ratio * advantage, 0.0) / len(s)
            grad = -coeff[:, None] * probs
            grad[np.arange(len(s)), a] += coeff
            np.add.at(logits, s, POLICY_LR * grad)

            # Gradient descent on 0.5 * mean((V(s) - GAE_target)^2).
            value_grad = (targets[indices] - values[s]) / len(s)
            np.add.at(values, s, VALUE_LR * value_grad)

    new_probs = softmax(logits[states])
    new_log_probs = np.log(np.maximum(new_probs[np.arange(n), actions], 1e-300))
    log_ratio = new_log_probs - old_log_probs
    ratio = np.exp(log_ratio)
    return {
        "approx_kl": float(np.mean((ratio - 1.0) - log_ratio)),
        "clip_fraction": float(np.mean(np.abs(ratio - 1.0) > CLIP_EPS)),
        "value_mse": float(np.mean((values[states] - targets) ** 2)),
    }


# Small, executable tests independent of FOOTSIES binaries.
a, target = compute_gae(
    rewards=[1.0, 2.0], old_values=[0.0, 0.0],
    next_values=[0.0, 0.0], terminals=[False, True],
)
np.testing.assert_allclose(a, [1 + GAMMA * GAE_LAMBDA * 2, 2])
np.testing.assert_allclose(target, a)

# A truncated final step bootstraps the critic, while a terminal step does not.
boot, _ = compute_gae([0.0], [0.0], [5.0], [False])
terminal, _ = compute_gae([0.0], [0.0], [5.0], [True])
np.testing.assert_allclose([boot[0], terminal[0]], [GAMMA * 5, 0.0])

# Positive advantage raises the sampled action probability; negative lowers it.
initial_good = softmax(logits[0])[3]
initial_bad = softmax(logits[0])[2]
example = {
    "states": np.array([0, 0]),
    "actions": np.array([3, 2]),
    "old_log_probs": np.full(2, np.log(1 / NUM_ACTIONS)),
    "advantages": np.array([1.0, -1.0]),
    "returns": np.array([1.0, -1.0]),
}
ppo_update([example], np.random.default_rng(0))
assert softmax(logits[0])[3] > initial_good
assert softmax(logits[0])[2] < initial_bad

# Verify that an overly large probability ratio is clipped for positive advantage.
logits[1, 3] = np.log(6.0)
clipped_example = {
    "states": np.array([1]),
    "actions": np.array([3]),
    "old_log_probs": np.array([np.log(1 / NUM_ACTIONS)]),
    "advantages": np.array([1.0]),
    "returns": np.array([1.0]),
}
before_clipped = logits.copy()
ppo_update([clipped_example], np.random.default_rng(1))
np.testing.assert_array_equal(logits, before_clipped)
print(f"GAE check: {a.round(3)}, truncated bootstrap: {boot[0]:.3f}")
print(f"PPO check: P(good) {initial_good:.3f} -> {softmax(logits[0])[3]:.3f}; "
      f"P(bad) {initial_bad:.3f} -> {softmax(logits[0])[2]:.3f}")

# Reset both tables after the tests.
logits[:] = 0.0
values[:] = 0.0

GAE check: [2.862 2.   ], truncated bootstrap: 4.900
PPO check: P(good) 0.167 -> 0.212; P(bad) 0.167 -> 0.128


In [3]:
def run_episode(env, mode="policy", seed=0, collect=False):
    """Play one episode; optionally collect a frozen-policy PPO trajectory."""
    if collect and mode != "policy":
        raise ValueError("Only on-policy episodes can be used for PPO updates")

    agent_rng = np.random.default_rng(seed + 100)
    opponent_rng = np.random.default_rng(seed + 200)
    obs, _ = env.reset(seed=seed)
    history = {k: [] for k in (
        "states", "actions", "old_log_probs", "rewards",
        "old_values", "next_values", "terminals"
    )}
    game_return, outcome = 0.0, 0

    for step in range(MAX_STEPS):
        state = encode_state(obs["p1"])
        if mode == "random":
            action = int(agent_rng.integers(NUM_ACTIONS))
        elif mode == "hardcoded":
            action = 2 if float(obs["p1"][0]) > 0.20 else 3
        elif mode == "policy":
            # Collect stochastically, evaluate greedily.
            action = choose_action(state, agent_rng, greedy=not collect)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        other = opponent_action(obs["p2"], opponent_rng)
        next_obs, step_rewards, terminated, truncated, _ = env.step({"p1": action, "p2": other})
        r = float(step_rewards["p1"])
        terminal = bool(terminated["p1"])
        done = terminal or bool(truncated["p1"])
        game_return += r

        if collect:
            self_loss = guard_health(obs["p1"]) - guard_health(next_obs["p1"])
            opp_loss = guard_health(obs["p1"], True) - guard_health(next_obs["p1"], True)
            shaped_reward = r + GUARD_REWARD * (opp_loss - self_loss)
            # old_log_probs and old_values must be captured BEFORE PPO updates.
            history["states"].append(state)
            history["actions"].append(action)
            history["old_log_probs"].append(float(np.log(softmax(logits[state])[action])))
            history["rewards"].append(shaped_reward)
            history["old_values"].append(float(values[state]))
            history["next_values"].append(float(values[encode_state(next_obs["p1"])]))
            history["terminals"].append(terminal)

        if done:
            if terminal:
                outcome = int(np.sign(r))
            break
        obs = next_obs

    metrics = (game_return, outcome, step + 1)
    if not collect:
        return metrics, None

    advantage, targets = compute_gae(
        history["rewards"], history["old_values"],
        history["next_values"], history["terminals"]
    )
    trajectory = {
        "states": np.asarray(history["states"], dtype=np.int64),
        "actions": np.asarray(history["actions"], dtype=np.int64),
        "old_log_probs": np.asarray(history["old_log_probs"], dtype=np.float64),
        "advantages": advantage,
        "returns": targets,
    }
    return metrics, trajectory


def evaluate(env, mode="policy", episodes=30, start_seed=10000):
    old_logits, old_values = logits.copy(), values.copy()
    results = [run_episode(env, mode, seed=start_seed + i)[0] for i in range(episodes)]
    assert np.array_equal(logits, old_logits), "Evaluation updated the actor!"
    assert np.array_equal(values, old_values), "Evaluation updated the critic!"
    returns, outcomes, lengths = np.asarray(results).T
    return {
        "return": float(returns.mean()),
        "win_rate": float(np.mean(outcomes == 1)),
        "loss_rate": float(np.mean(outcomes == -1)),
        "avg_steps": float(lengths.mean()),
    }

In [4]:
# FOOTSIES is only needed for the actual game, not for the math checks above.
import footsiesgym
from footsiesgym.footsies.game import constants

ACTION_NAMES = list(constants.FOOTSIES_ACTION_IDS)
ATTACK_INDICES = {
    ACTION_NAMES.index(name)
    for name in ("N_ATTACK", "B_ATTACK", "N_SPECIAL", "B_SPECIAL")
}

env = footsiesgym.make(
    config={"port": PORT, "headless": True, "max_t": MAX_STEPS, "action_delay": 0},
    platform=PLATFORM,
    launch_binaries=LAUNCH_BINARIES,
)

try:
    assert env.action_space("p1").n == NUM_ACTIONS
    obs, _ = env.reset()
    assert len(obs["p1"]) == 88
    print("Actor shape:", logits.shape, "| critic shape:", values.shape)

    rng = np.random.default_rng(42)
    batch = []
    best_return, best_policy, best_values = -float("inf"), logits.copy(), values.copy()
    diagnostics = {}

    for ep in range(1, TRAIN_EPISODES + 1):
        _, trajectory = run_episode(env, seed=ep, collect=True)
        batch.append(trajectory)

        # Update only after collecting a complete on-policy batch.
        if len(batch) == BATCH_EPISODES or ep == TRAIN_EPISODES:
            diagnostics = ppo_update(batch, rng)
            batch.clear()

        if ep % EVAL_EVERY == 0:
            stats = evaluate(env, episodes=QUICK_EVAL_EPISODES)
            if stats["return"] > best_return:
                best_return = stats["return"]
                best_policy, best_values = logits.copy(), values.copy()
            print(f"Episode {ep:3d} | eval return={stats['return']:+.3f} | "
                  f"win={stats['win_rate']:.1%} | "
                  f"approx KL={diagnostics['approx_kl']:.4f} | "
                  f"clip fraction={diagnostics['clip_fraction']:.1%}")

    logits[:], values[:] = best_policy, best_values
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    path = CHECKPOINT_DIR / "PPO_GAE_best.npz"
    np.savez_compressed(path, logits=logits, values=values)

    print("\nFinal evaluation (same opponent and seeds):")
    for mode in ("random", "hardcoded", "policy"):
        stats = evaluate(env, mode, FINAL_EVAL_EPISODES, start_seed=20000)
        print(f"{mode:9s} | return={stats['return']:+.3f} | "
              f"win={stats['win_rate']:.1%} | loss={stats['loss_rate']:.1%}")
    print("Saved:", path)
finally:
    env.close()

Actor shape: (12, 6) | critic shape: (12,)
Episode  50 | eval return=-0.200 | win=26.7% | approx KL=0.0018 | clip fraction=0.0%
Episode 100 | eval return=-0.367 | win=16.7% | approx KL=0.0015 | clip fraction=0.0%
Episode 150 | eval return=-1.000 | win=0.0% | approx KL=0.0031 | clip fraction=0.0%
Episode 200 | eval return=-1.000 | win=0.0% | approx KL=0.0008 | clip fraction=0.0%
Episode 250 | eval return=-0.933 | win=3.3% | approx KL=0.0015 | clip fraction=0.0%
Episode 300 | eval return=+0.933 | win=96.7% | approx KL=0.0007 | clip fraction=0.0%

Final evaluation (same opponent and seeds):
random    | return=+0.067 | win=53.3% | loss=46.7%
hardcoded | return=-1.000 | win=0.0% | loss=100.0%
policy    | return=+0.867 | win=93.3% | loss=6.7%
Saved: footsies_checkpoints/PPO_GAE_best.npz


## Comparison with the original REINFORCE notebook

- **Same game setup:** 12 discrete states, 6 actions, scripted opponent, guard-reward shaping, and raw game-return evaluation. No new model dependencies beyond NumPy.
- **REINFORCE:** one Monte Carlo return-to-go per step; one policy update after every episode; no critic.
- **PPO + GAE:** state-value baseline, bootstrapped temporal-difference residuals, GAE($\lambda$), stored old action log-probabilities, clipped policy objective, multiple epochs over each on-policy batch, and critic regression.
- **Evaluation fix:** `policy` now selects the argmax action at evaluation, rather than sampling. Training still samples. Because this changes evaluation semantics, earlier stochastic-policy results are not directly comparable.
- **Truncation:** both environment time limits and the local `MAX_STEPS` loop limit bootstrap the critic if the environment did not truly terminate. Each rollout is processed independently, so GAE cannot cross a reset.

The critic is trained on **shaped** rewards while the printed game return remains **unshaped**. A 12-state observation aliasing many distinct game situations limits what either actor or critic can learn. PPO is not guaranteed to outperform REINFORCE in 300 episodes; tune the batch size, step sizes, and reward scale if training plateaus.

**Run requirements:** install `numpy` and `footsies-gym`; start the macOS game server on port `50051` (as for the original notebook), or adjust the platform/port settings. The small tests above run without FOOTSIES. The checkpoint stores both tables in `PPO_GAE_best.npz`.